## Setup

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# !pip install -q -U langchain langchain-core langchain-community langchain-text-splitters langchain-ollama langgraph "requests==2.32.4" chromadb


## Imports & Config

In [3]:
import json
import pickle
import time
import uuid
from collections import defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import chromadb
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama

DATA_DIR   = Path("data")
CACHE_DIR  = Path("cache")
VECTOR_DIR = Path("vectorDB")

for d in (DATA_DIR, CACHE_DIR, VECTOR_DIR):
    d.mkdir(exist_ok=True)

EMBED_MODEL    = "nomic-embed-text:v1.5"
CLASSIFY_MODEL = "granite4.2:8b"
GENERATE_MODEL = "qwen2.5:3b"
JUDGE_MODEL    = "granite4.2:8b"


/Users/rishabhh/Yash/capstone/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## PDF Loading

In [4]:
def process_all_pdf(pdf_directory: Path):
    all_documents = []
    pdf_files = list(pdf_directory.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")
    for pdf_file in pdf_files:
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(documents)
            print(f"  Loaded {len(documents)} pages — {pdf_file.name}")
        except Exception as e:
            print(f"  Error {pdf_file.name}: {e}")
    print(f"Total pages loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdf(DATA_DIR)


Found 15 PDF files
  Loaded 13 pages — Weather_Insurance_Policy_Wordings_Retail_6a1bd806a7.pdf
  Loaded 43 pages — Click-2-Protect-Optima-Secure-Policy-Bond-101Y122V05.pdf
  Loaded 42 pages — hdfc-life-smart-pension-plus-v13-policy-document-individual.pdf
  Loaded 11 pages — Cyber_Shield_Policy_Wordings_78baa23b5a.pdf
  Loaded 5 pages — SBI_General_Livestock_Policy_Wording_38ef0b201b.pdf
  Loaded 47 pages — Auto_Secure_Commercial_Vehicle_Package_Policy_Base_Policy_Wording_22b7905015.pdf
  Loaded 11 pages — Policy_Wordings_contractors_plant_and_machinery_insurance.pdf_0175bcd067.pdf
  Loaded 21 pages — Bharat_Griha_Raksha_Policy_Policy_Wordings_5219f40e18.pdf
  Loaded 18 pages — Policy_Wordings_aviation_insurance.pdf_7b7e60200a.pdf
  Loaded 62 pages — tata_aig_travel_insurance_international_plus_health_policy_wordings_012c2ec139.pdf
  Loaded 23 pages — Policy_Wordings_political_risk_insurance_for_investors.pdf_b7a0e7c805.pdf
  Loaded 20 pages — 14..Trade credit insc_GEN756.pdf
  Loaded 

## Text Splitting

In [5]:
def split_documents(documents, chunk_size=1000, chunk_overlap=150, min_chunk_size=80):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n\n", "\n\n", "\n", ". ", "? ", "! ", "; ", ", ", " ", ""],
        length_function=len,
        is_separator_regex=False,
        keep_separator=True,
    )
    split_docs = splitter.split_documents(documents)
    cleaned = []
    for doc in split_docs:
        text = doc.page_content.strip()
        if len(text) < min_chunk_size:
            continue
        alnum = sum(c.isalnum() for c in text)
        if alnum / max(len(text), 1) < 0.3:
            continue
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        if len(lines) == 1 and any(lines[0].lower().startswith(p) for p in ["page ", "www.", "copyright"]):
            continue
        doc.page_content = text
        cleaned.append(doc)
    for i, doc in enumerate(cleaned):
        doc.metadata["chunk_index"] = i
        doc.metadata["chunk_size"] = len(doc.page_content)
        doc.metadata["chunk_word_count"] = len(doc.page_content.split())
    print(f"Chunks after filtering: {len(cleaned)}")
    return cleaned


## Checkpoint — Chunks

In [6]:
chunks_file = CACHE_DIR / "chunks.pkl"

if chunks_file.exists():
    with open(chunks_file, "rb") as f:
        chunks = pickle.load(f)
    print(f"✓ Loaded {len(chunks)} chunks from cache")
else:
    chunks = split_documents(all_pdf_documents)
    with open(chunks_file, "wb") as f:
        pickle.dump(chunks, f)
    print(f"✓ Saved {len(chunks)} chunks to cache")


✓ Loaded 2096 chunks from cache


## Embedding (nomic-embed-text via Ollama)

In [7]:
class EmbeddingManager:
    def __init__(self, model_name: str = EMBED_MODEL):
        self.model_name = model_name
        self._dim: int | None = None
        self.model = OllamaEmbeddings(model=model_name)
        print(f"Embedding model ready: {model_name}")

    def generate_embeddings(self, texts: list[str], batch_size: int = 20) -> np.ndarray:
        all_embeddings = []
        total = len(texts)
        print(f"Embedding {total} texts in batches of {batch_size}…")
        for i in range(0, total, batch_size):
            batch = texts[i : i + batch_size]
            for attempt in range(3):
                try:
                    result = self.model.embed_documents(batch)
                    all_embeddings.extend(result)
                    print(f"  Batch {i // batch_size + 1}/{(total + batch_size - 1) // batch_size} done")
                    break
                except Exception as e:
                    wait = 2 ** attempt
                    print(f"  Attempt {attempt + 1} failed: {e}. Retrying in {wait}s…")
                    time.sleep(wait)
            else:
                raise RuntimeError(f"Batch {i // batch_size + 1} failed after 3 attempts")
        return np.asarray(all_embeddings)

    def embed_query(self, query: str) -> np.ndarray:
        return np.asarray(self.model.embed_query(query))

    def get_dim(self) -> int:
        if self._dim is None:
            self._dim = len(self.model.embed_query("probe"))
        return self._dim

embedding_manager = EmbeddingManager()


Embedding model ready: nomic-embed-text:v1.5


## Checkpoint — Embeddings

In [8]:
embeddings_file = CACHE_DIR / "embeddings.npy"

if embeddings_file.exists():
    embeddings = np.load(embeddings_file)
    print(f"✓ Loaded embeddings: {embeddings.shape}")
else:
    embeddings = embedding_manager.generate_embeddings([doc.page_content for doc in chunks])
    np.save(embeddings_file, embeddings)
    print(f"✓ Saved embeddings: {embeddings.shape}")


✓ Loaded embeddings: (2096, 768)


## Vector Store (ChromaDB)

In [9]:
class VectorStore:
    def __init__(self, collection_name: str, persist_directory: str = str(VECTOR_DIR)):
        self.collection_name = collection_name
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"description": "PDF Document embeddings for RAG", "hnsw:space": "cosine"},
        )
        print(f"Vector store ready — {self.collection.count()} docs in '{collection_name}'")

    def add_documents(self, documents: list[Any], embeddings: np.ndarray):
        ids, metadatas, texts, vecs = [], [], [], []
        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            ids.append(f"doc_{uuid.uuid4().hex[:8]}_{i}")
            meta = dict(doc.metadata)
            meta["doc_index"] = i
            meta["content_length"] = len(doc.page_content)
            metadatas.append(meta)
            texts.append(doc.page_content)
            vecs.append(emb.tolist())
        batch_size = 1000
        for start in range(0, len(documents), batch_size):
            end = start + batch_size
            self.collection.add(
                ids=ids[start:end],
                embeddings=vecs[start:end],
                metadatas=metadatas[start:end],
                documents=texts[start:end],
            )
        print(f"✓ Added {len(documents)} docs — total: {self.collection.count()}")


## Checkpoint — Vector DB Population

In [10]:
vector_store = VectorStore(collection_name="policy_documents")

if vector_store.collection.count() == 0:
    vector_store.add_documents(embeddings=embeddings, documents=chunks)
    print("✓ Vector DB populated")
else:
    print(f"✓ Vector DB already has {vector_store.collection.count()} docs — skipping")


Vector store ready — 2096 docs in 'policy_documents'
✓ Vector DB already has 2096 docs — skipping


## RAG Retriever

In [11]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> list[dict]:
        try:
            qvec = self.embedding_manager.embed_query(query)
            results = self.vector_store.collection.query(
                query_embeddings=[qvec.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances"],
            )
            retrieved = []
            if results["documents"] and results["documents"][0]:
                for i, (doc_id, doc, meta, dist) in enumerate(zip(
                    results["ids"][0], results["documents"][0],
                    results["metadatas"][0], results["distances"][0],
                )):
                    score = 1 - dist
                    if score >= score_threshold:
                        retrieved.append({"id": doc_id, "content": doc, "metadata": meta,
                                          "similarity_score": score, "rank": i + 1})
            return retrieved
        except Exception as e:
            print(f"Retrieval error: {e}")
            return []

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)


## LLMs (Ollama)

In [12]:
llm          = ChatOllama(model=GENERATE_MODEL, temperature=0.0)
classify_llm = ChatOllama(model=CLASSIFY_MODEL, temperature=0.0)


## Query Pipeline

In [13]:
GENERAL_SYSTEM_PROMPT = """You are an expert insurance advisor.
Answer the following general insurance question clearly and helpfully.
Use your general knowledge — you do NOT need to reference any specific policy document.

Question: {query}

Answer:"""

SPECIFIC_SYSTEM_PROMPT = """You are an expert insurance policy assistant. Answer the question using ONLY the provided context from the actual policy documents.

Rules:
- Use the context to provide a direct, factual answer
- If the context partially answers the question, provide what you can and note what is missing
- Only say "I don't have enough information" if the context is completely irrelevant to the question
- Be concise and precise
- Do not hallucinate or add information not present in the context"""

CLASSIFIER_PROMPT = """You are a query classifier for an insurance policy assistant.

Classify the query into exactly one of these categories:
- general: insurance-related conceptual, definitional, or comparative questions
- specific: the user asks about a concrete provision, condition, claim, or coverage detail of a policy
- ambiguous: too vague to route
- irrelevant: NOT related to insurance, policies, coverage, health plans, or financial protection

Respond with ONLY the category word (general, specific, ambiguous, or irrelevant), nothing else.

Query: {query}
Category:"""


def classify_query(query: str, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            response = classify_llm.invoke(CLASSIFIER_PROMPT.format(query=query))
            category = response.content.strip().lower()
            return category if category in ("general", "specific", "ambiguous", "irrelevant") else "specific"
        except Exception:
            if attempt < retries - 1:
                time.sleep(2)
    return "specific"


def rag_query(query: str, top_k: int = 5, score_threshold: float = 0.2) -> dict:
    if not query or not query.strip():
        return {"answer": "Query cannot be empty", "sources": [], "context_used": False}

    category = classify_query(query)

    if category == "irrelevant":
        return {"answer": "I'm an insurance policy assistant and can only help with questions related to insurance.",
                "sources": [], "context_used": False, "category": category, "confidence": 1.0}

    if category == "general":
        for attempt in range(3):
            try:
                response = llm.invoke(GENERAL_SYSTEM_PROMPT.format(query=query))
                return {"answer": response.content, "sources": [], "context_used": False,
                        "category": category, "confidence": 1.0}
            except Exception:
                if attempt < 2:
                    time.sleep(3)
        return {"answer": "Generation failed.", "sources": [], "context_used": False,
                "category": category, "confidence": 0.0}

    if category == "ambiguous":
        return {"answer": "Your question is too vague. Could you specify which policy or coverage detail you're asking about?",
                "sources": [], "context_used": False, "category": category}

    results = rag_retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)
    if not results:
        return {"answer": "I don't have enough information in the provided context to answer this.",
                "sources": [], "context_used": False, "category": category, "confidence": 0.0}

    context_parts, sources = [], []
    for i, doc in enumerate(results):
        context_parts.append(f"[{i + 1}] {doc['content']}")
        sources.append({"id": doc["id"], "score": round(doc["similarity_score"], 4),
                        "page": doc["metadata"].get("page", "unknown"),
                        "source": doc["metadata"].get("source_file", "unknown"),
                        "preview": doc["content"][:150].strip() + "…"})

    confidence = max(doc["similarity_score"] for doc in results)
    prompt = f"{SPECIFIC_SYSTEM_PROMPT}\n\nContext: {chr(10).join(context_parts)}\n\nQuestion: {query}\n\nAnswer:"

    for attempt in range(3):
        try:
            response = llm.invoke(prompt)
            return {"answer": response.content, "sources": sources, "context_used": True,
                    "retrieved_count": len(results), "category": category, "confidence": confidence}
        except Exception:
            if attempt < 2:
                time.sleep(3)
    return {"answer": "Generation timed out.", "sources": sources, "context_used": False,
            "category": category, "confidence": 0.0}


## Quick Test

In [14]:
# for q in [
#     "Why should I have a health insurance?",
#     "Write me a python program for Hello World",
#     "Under what timeframe must the Insured submit a complete written claim?",
# ]:
#     print(f"Q: {q}")
#     out = rag_query(q, top_k=3)
#     for k, v in out.items():
#         print(f"  {k}: {v}")
#     print()


## RAG Evaluation — Load policy_qa.json

In [15]:
QA_FILE = DATA_DIR / "policy_qa.json"
with open(QA_FILE, encoding="utf-8") as f:
    test_set = json.load(f)
print(f"✓ Loaded {len(test_set)} QA pairs from {QA_FILE}")


✓ Loaded 83 QA pairs from data/policy_qa.json


## Checkpoint — Vector RAG Outputs (83 questions)

In [16]:
rag_outputs_file = CACHE_DIR / "vector_rag_outputs.json"

def run_vector_rag_pipeline(test_set: list[dict], checkpoint_file: Path) -> list[dict]:
    # Load existing progress
    if checkpoint_file.exists():
        with open(checkpoint_file) as f:
            outputs = json.load(f)
        done_qs = {o["question"] for o in outputs}
        print(f"Resuming from checkpoint — {len(outputs)}/{len(test_set)} done")
    else:
        outputs = []
        done_qs = set()

    remaining = [item for item in test_set if item["question"] not in done_qs]
    print(f"{len(remaining)} questions remaining")

    for i, item in enumerate(remaining, 1):
        q = item["question"]
        print(f"[{len(outputs) + 1}/{len(test_set)}] {q[:70]}…")
        try:
            retrieved = rag_retriever.retrieve(q, top_k=5, score_threshold=0.2)
            contexts = [doc["content"] for doc in retrieved]
            result = rag_query(query=q, top_k=5)
            outputs.append({
                "question": q,
                "ground_truth": item["ground_truth"],
                "contexts": contexts,
                "answer": result.get("answer", ""),
                "category": result.get("category", "RELEVANT"),
                "source_file": item.get("source_file", ""),
            })
            print(f"  ✓ {outputs[-1]['answer'][:80]}…")
        except Exception as e:
            print(f"  ✗ Error: {e}")
            outputs.append({
                "question": q,
                "ground_truth": item.get("ground_truth", ""),
                "contexts": [],
                "answer": f"ERROR: {e}",
                "category": "ERROR",
                "source_file": item.get("source_file", ""),
            })

        # Save checkpoint every question
        with open(checkpoint_file, "w") as f:
            json.dump(outputs, f, indent=2)

    print(f"\n✓ Done — {len(outputs)} outputs saved to {checkpoint_file}")
    return outputs


if rag_outputs_file.exists():
    with open(rag_outputs_file) as f:
        vector_rag_outputs = json.load(f)
    if len(vector_rag_outputs) < len(test_set):
        print(f"Incomplete checkpoint ({len(vector_rag_outputs)}/{len(test_set)}) — resuming")
        vector_rag_outputs = run_vector_rag_pipeline(test_set, rag_outputs_file)
    else:
        print(f"✓ Loaded complete RAG outputs ({len(vector_rag_outputs)} entries)")
else:
    vector_rag_outputs = run_vector_rag_pipeline(test_set, rag_outputs_file)


✓ Loaded complete RAG outputs (83 entries)


## Evaluation Prompts

In [17]:
FAITHFULNESS_PROMPT = """You are an expert evaluator assessing whether an AI answer is faithful to its source context.

QUESTION: {question}

RETRIEVED CONTEXT:
{context}

GENERATED ANSWER:
{answer}

TASK:
1. List every factual claim made in the Generated Answer.
2. For each claim, determine if it is directly supported by the Retrieved Context.
3. Score = (number of supported claims) / (total claims). If there are no claims, score 1.0.

Respond in this EXACT JSON format only (no markdown, no extra text):
{{"score": 0.0, "reasoning": "claim 1: supported/not supported because... claim 2: ..."}}

Score must be between 0.0 and 1.0."""

ANSWER_RELEVANCY_PROMPT = """You are an expert evaluator assessing whether an AI answer is relevant to the question asked.

QUESTION: {question}

GENERATED ANSWER:
{answer}

TASK:
Score how directly and completely the answer addresses the question.
- 1.0 = answer directly addresses all parts of the question
- 0.7 = answer mostly relevant but misses a part or adds off-topic content
- 0.4 = answer is vaguely related but does not really answer the question
- 0.0 = answer is completely off-topic or refuses to answer

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "explanation of why this score was given"}}"""

CONTEXT_PRECISION_PROMPT = """You are an expert evaluator assessing the quality of retrieved context for a RAG system.

QUESTION: {question}

GROUND TRUTH ANSWER:
{ground_truth}

RETRIEVED CONTEXT CHUNKS:
{context_numbered}

TASK:
For each retrieved chunk, decide if it is relevant to answering the question (given what the ground truth says).
Score = (number of relevant chunks) / (total chunks).
If no chunks were retrieved, score is 0.0.

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "Chunk 1: relevant/not relevant because... Chunk 2: ..."}}"""

CONTEXT_RECALL_PROMPT = """You are an expert evaluator assessing whether a RAG system retrieved all necessary information.

QUESTION: {question}

GROUND TRUTH ANSWER:
{ground_truth}

RETRIEVED CONTEXT:
{context}

TASK:
1. List every key piece of information in the Ground Truth Answer.
2. For each key piece, check if it is present in the Retrieved Context.
3. Score = (pieces present in context) / (total key pieces).
If no context was retrieved, score is 0.0.

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "Key point 1: found/not found in context... Key point 2: ..."}}"""


## Evaluation Functions

In [18]:
JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "score": {"type": "number"},
        "reasoning": {"type": "string"},
    },
    "required": ["score", "reasoning"],
}

judge_llm = ChatOllama(model=JUDGE_MODEL, temperature=0.0, format=JUDGE_SCHEMA, reasoning=False)


def _try_parse(raw: str) -> dict | None:
    try:
        clean = raw.replace("```json", "").replace("```", "").strip()
        start, end = clean.find("{"), clean.rfind("}") + 1
        if start == -1 or end == 0:
            return None
        parsed = json.loads(clean[start:end])
        score = float(parsed.get("score", 0.0))
        return {"score": max(0.0, min(1.0, score)), "reasoning": str(parsed.get("reasoning", ""))}
    except Exception:
        return None


def _regex_fallback(raw: str) -> dict | None:
    import re
    m = re.search(r'"?score"?\s*[:=]\s*(-?\d*\.?\d+)', raw, re.IGNORECASE)
    if not m:
        return None
    try:
        score = float(m.group(1))
        return {"score": max(0.0, min(1.0, score)), "reasoning": raw.strip()[:500]}
    except Exception:
        return None


def judge_score(prompt: str, max_retries: int = 2) -> dict:
    """Invoke the judge LLM and parse a {score, reasoning} dict.

    With format="json" enforced on judge_llm, the first attempt almost
    always parses cleanly, so this stays a short, cheap safety net (not
    a long backoff loop) rather than something the happy path pays for.
    """
    last_raw = ""
    for attempt in range(max_retries):
        try:
            response = judge_llm.invoke(prompt)
            raw = response.content.strip()
            last_raw = raw
        except Exception as e:
            print(f"  Judge attempt {attempt + 1} failed: {e}. Retrying in 2s…")
            time.sleep(2)
            continue

        parsed = _try_parse(raw)
        if parsed is not None:
            return parsed

        if attempt < max_retries - 1:
            print(f"  Judge attempt {attempt + 1} returned unparseable output. Retrying…")

    # Last resort: pull a bare numeric score out of whatever text we got.
    fallback = _regex_fallback(last_raw)
    if fallback is not None:
        return fallback
    return {"score": 0.0, "reasoning": f"Parse error after {max_retries} attempts | Raw: {last_raw[:200]}"}


def evaluate_single(item: dict) -> dict:
    q, a, gt = item["question"], item["answer"], item["ground_truth"]
    ctx = item.get("contexts", [])
    context_joined = "\n\n".join(ctx) if ctx else "[No context retrieved]"
    context_numbered = (
        "\n\n".join(f"[Chunk {i + 1}]:\n{c}" for i, c in enumerate(ctx))
        if ctx else "[No context retrieved]"
    )
    return {
        "faithfulness":      judge_score(FAITHFULNESS_PROMPT.format(question=q, context=context_joined, answer=a)),
        "answer_relevancy":  judge_score(ANSWER_RELEVANCY_PROMPT.format(question=q, answer=a)),
        "context_precision": judge_score(CONTEXT_PRECISION_PROMPT.format(question=q, ground_truth=gt, context_numbered=context_numbered)),
        "context_recall":    judge_score(CONTEXT_RECALL_PROMPT.format(question=q, ground_truth=gt, context=context_joined)),
    }

## Checkpoint — Evaluation (83 × 4 judge calls)

In [19]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

eval_file = CACHE_DIR / "vector_rag_eval_results.json"

EVAL_WORKERS = 2  # matches OLLAMA_NUM_PARALLEL — judge calls for different
                  # questions are independent, so running 2 concurrently
                  # overlaps them at the Ollama server without changing
                  # any prompt, model, or score computation.


def evaluate_pipeline(pipeline_outputs: list[dict], checkpoint_file: Path) -> list[dict]:
    if checkpoint_file.exists():
        with open(checkpoint_file) as f:
            results = json.load(f)
        done_qs = {r["question"] for r in results}
        print(f"Resuming eval — {len(results)}/{len(pipeline_outputs)} done")
    else:
        results = []
        done_qs = set()

    remaining = [item for item in pipeline_outputs
                 if item["question"] not in done_qs
                 and not item["answer"].startswith("ERROR")]

    print(f"{len(remaining)} questions left to evaluate ({EVAL_WORKERS} concurrent)")

    save_lock = threading.Lock()

    def run_one(item: dict) -> dict:
        scores = evaluate_single(item)
        return {
            "question":          item["question"],
            "answer":            item["answer"],
            "ground_truth":      item["ground_truth"],
            "n_contexts":        len(item.get("contexts", [])),
            "faithfulness":      scores["faithfulness"]["score"],
            "answer_relevancy":  scores["answer_relevancy"]["score"],
            "context_precision": scores["context_precision"]["score"],
            "context_recall":    scores["context_recall"]["score"],
            "reasoning":         scores,
        }

    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        futures = {pool.submit(run_one, item): item for item in remaining}
        for future in as_completed(futures):
            item = futures[future]
            r = future.result()
            with save_lock:
                results.append(r)
                print(f"[{len(results)}/{len(pipeline_outputs)}] {r['question'][:65]}…")
                print(f"  F={r['faithfulness']:.2f}  AR={r['answer_relevancy']:.2f}  "
                      f"CP={r['context_precision']:.2f}  CR={r['context_recall']:.2f}")
                with open(checkpoint_file, "w") as f:
                    json.dump(results, f, indent=2)

    print(f"\n✓ Evaluation complete — {len(results)} results in {checkpoint_file}")
    return results


if eval_file.exists():
    with open(eval_file) as f:
        eval_results = json.load(f)
    if len(eval_results) < len(vector_rag_outputs):
        print(f"Incomplete eval ({len(eval_results)}/{len(vector_rag_outputs)}) — resuming")
        eval_results = evaluate_pipeline(vector_rag_outputs, eval_file)
    else:
        print(f"✓ Loaded complete eval results ({len(eval_results)} entries)")
else:
    eval_results = evaluate_pipeline(vector_rag_outputs, eval_file)


Incomplete eval (31/83) — resuming
Resuming eval — 31/83 done
51 questions left to evaluate (2 concurrent)
[32/83] What types of property and specific items are included as covered…
  F=1.00  AR=0.70  CP=0.20  CR=1.00
[33/83] Which accounts receivable are excluded from the claim amount unde…
  F=1.00  AR=0.00  CP=1.00  CR=0.00
[34/83] How do I file a claim under this policy and what documents are re…
  F=1.00  AR=0.70  CP=0.00  CR=0.00
[35/83] What is the market value limit for equipment covered under the me…
  F=1.00  AR=0.40  CP=0.20  CR=1.00
[36/83] What does the policy state regarding electronic transactions?…
  F=1.00  AR=0.00  CP=0.00  CR=0.00
[37/83] What is the maximum duration of the indemnity period under this p…
  F=0.00  AR=0.70  CP=1.00  CR=1.00
[38/83] What types of coverage or parties are specifically mentioned as b…
  F=0.86  AR=0.70  CP=0.00  CR=0.00
[39/83] According to the policy, what is the maximum amount for which the…
  F=1.00  AR=0.40  CP=0.60  CR=1.00
[40/83] A

In [20]:
# import subprocess
# import time
# import requests

# # Start Ollama daemon as a persistent process
# subprocess.Popen(["ollama", "serve"])
# time.sleep(4)

# # Health check
# try:
#     res = requests.get("http://127.0.0.1:11434/api/tags")
#     print("Ollama Status:", res.status_code, "Models:", [m["name"] for m in res.json().get("models", [])])
# except Exception as e:
#     print("Failed to connect:", e)

In [21]:
# !pkill -9 -f ollama
# !nohup ollama serve > ollama.log 2>&1 &
# !sleep 3
# !curl -s http://127.0.0.1:11434/api/tags

## Results Summary

In [23]:
import pandas as pd

METRICS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
df = pd.DataFrame(eval_results)
avgs = df[METRICS].mean().round(3)

print("=" * 65)
print("  VECTOR RAG EVALUATION RESULTS")
print("=" * 65)
print(avgs.to_string())
print("=" * 65)
print("  All scores 0.0 – 1.0  |  Higher is better")
print("  F=Faithfulness | AR=Answer Relevancy | CP=Context Precision | CR=Context Recall")


  VECTOR RAG EVALUATION RESULTS
faithfulness         0.648
answer_relevancy     0.645
context_precision    0.307
context_recall       0.376
  All scores 0.0 – 1.0  |  Higher is better
  F=Faithfulness | AR=Answer Relevancy | CP=Context Precision | CR=Context Recall
